# Ablation Study: Impact of Backbone Freezing on Re-ID Performance

We compare 3 fine-tuning strategies on ResNet-50 / Market-1501:
1. **Feature Extraction** — Backbone fully frozen, only BN-Neck + classifier trained
2. **Partial Fine-tuning** — Early layers frozen (conv1→layer2), layer3/4 + head trained
3. **Full Fine-tuning** — Entire network trained (our baseline)

In [ ]:
import os, sys, yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from torchvision import transforms

In [ ]:
if os.getcwd().endswith('notebooks'):
    os.chdir('..')
sys.path.append(os.getcwd())

In [ ]:
from src.dataloaders.market_dataset import MarketDataset
from src.models.resnet50 import ResNet50
from src.utils.losses import TripletLoss
from src.utils.trainer import train_model
from src.utils.evaluator import extract_features, evaluate

In [ ]:
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)['reid_project']

m_cfg = config['models']['resnet50']
device = torch.device(config['general']['device'])
print(f"Device: {device}")

In [ ]:
transform = transforms.Compose([
    transforms.Resize(config['market1501']['img_size'],
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = MarketDataset(root_dir=config['market1501']['root_dir'], subset='train', transform=transform)
query_ds = MarketDataset(root_dir=config['market1501']['root_dir'], subset='query', transform=transform)
gallery_ds = MarketDataset(root_dir=config['market1501']['root_dir'], subset='test', transform=transform)

train_loader = DataLoader(train_ds, batch_size=m_cfg['train']['batch_size'], shuffle=True)
query_loader = DataLoader(query_ds, batch_size=32, shuffle=False)
gallery_loader = DataLoader(gallery_ds, batch_size=32, shuffle=False)

## Training & Evaluation Loop for each strategy

In [ ]:
FREEZE_MODES = ['feature_extraction', 'partial', 'full']
NUM_EPOCHS = m_cfg['train']['epochs']

results = {}  # {mode: {'history': ..., 'rank1': ..., 'mAP': ...}}

In [ ]:
for mode in FREEZE_MODES:
    print(f"\n{'='*60}")
    print(f"  Strategy: {mode}")
    print(f"{'='*60}")
    
    # Fresh model each time (ImageNet pretrained)
    model = ResNet50(
        num_classes=config['market1501']['num_classes'],
        feature_dim=m_cfg['feature_dim'],
        last_stride=m_cfg['last_stride']
    )
    model.freeze_backbone(mode=mode)
    model = model.to(device)
    
    # Only optimize trainable params
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=m_cfg['train']['learning_rate']
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    criterion_xent = torch.nn.CrossEntropyLoss()
    criterion_triplet = TripletLoss(margin=m_cfg['loss']['margin'])
    
    output_dir = f"results/ablation_{mode}"
    
    # Train
    history = train_model(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion_xent=criterion_xent,
        criterion_triplet=criterion_triplet,
        device=device,
        config=m_cfg['loss'],
        num_epochs=NUM_EPOCHS,
        output_dir=output_dir
    )
    
    # Evaluate
    q_f, q_p, q_c = extract_features(model, query_loader, device)
    g_f, g_p, g_c = extract_features(model, gallery_loader, device)
    rank1, mAP, _ = evaluate(q_f, q_p, g_f, g_p, q_c, g_c)
    
    results[mode] = {
        'history': history,
        'rank1': rank1,
        'mAP': mAP
    }
    
    print(f"\n>>> {mode}: Rank-1 = {rank1*100:.2f}%, mAP = {mAP*100:.2f}%")

## Results Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: Loss curves ---
ax = axes[0]
for mode, data in results.items():
    ax.plot(data['history']['epoch'], data['history']['loss'], marker='o', label=mode)
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Loss Convergence by Freezing Strategy')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Plot 2: Rank-1 bar chart ---
ax = axes[1]
modes = list(results.keys())
rank1_vals = [results[m]['rank1'] * 100 for m in modes]
colors = ['#3498db', '#f39c12', '#2ecc71']
bars = ax.bar(modes, rank1_vals, color=colors)
ax.set_ylabel('Rank-1 Accuracy (%)')
ax.set_title('Rank-1 by Strategy')
ax.set_ylim(0, 100)
for bar, val in zip(bars, rank1_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', fontweight='bold')

# --- Plot 3: mAP bar chart ---
ax = axes[2]
mAP_vals = [results[m]['mAP'] * 100 for m in modes]
bars = ax.bar(modes, mAP_vals, color=colors)
ax.set_ylabel('mAP (%)')
ax.set_title('mAP by Strategy')
ax.set_ylim(0, 100)
for bar, val in zip(bars, mAP_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('results/ablation_freezing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to results/ablation_freezing_comparison.png")

In [ ]:
# Summary table
print(f"{'Strategy':<25} {'Rank-1':>8} {'mAP':>8} {'Final Loss':>12}")
print('-' * 55)
for mode, data in results.items():
    print(f"{mode:<25} {data['rank1']*100:>7.2f}% {data['mAP']*100:>7.2f}% {data['history']['loss'][-1]:>11.4f}")